In [ ]:
"""main_v3_dual_evaluator.py

LangGraph Joke Generation Pipeline with Dual Evaluation (GPT-OSS + Mistral)
"""

# !pip uninstall -y transformers accelerate peft bitsandbytes trl
# !pip install -q -U bitsandbytes accelerate peft transformers trl datasets
# !pip install faiss-cpu

import os
import json
import torch
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

import wandb
from huggingface_hub import notebook_login

In [ ]:
"""
Setup environment and imports
"""

print("Setting up environment...")

# Check for GPU
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")
else:
    print("No GPU detected. Will use CPU (slower)")

print(" Environment setup complete\n")

"""
LangGraph Joke Generation Pipeline - Orchestrator with Dual Evaluation
Imports and orchestrates existing implementations
"""

"""
LangGraph Joke Generation Pipeline - Orchestrator
Imports and orchestrates existing implementations
"""
from typing import TypedDict, Literal, List, Dict
from langgraph.graph import StateGraph, END
from pathlib import Path

import sys

#  set project root = folder c:\Users\Anwender\humor-project
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# if you run from a notebook inside some subfolder, walk up until we find "src"
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))         # so "import src...." works
sys.path.insert(0, str(PROJECT_ROOT / "src")) # so "import evaluation...." works if you want
print("Using PROJECT_ROOT:", PROJECT_ROOT)
# Import your existing implementations
from using_llama_with_rag import RAGHumorGenerator, RAGConfig
from gpt_joke_generation import GPT4JokeGenerator
from src.evaluation.llm_evaluator import LLMHumorScorer
from src.evaluation.mistral_evaluator import MistralHumorScorer

print(" All modules imported successfully\n")

In [ ]:

# ============================================================================
# LANGGRAPH STATE DEFINITION
# ============================================================================

class JokeGenerationState(TypedDict):
    """State for the joke generation pipeline"""
    # Input
    word1: str
    word2: str

    # Generation tracking
    llama_jokes: List[str]
    gpt_jokes: List[str]
    candidate_jokes: List[str]
    joke_sources: List[str]      # "Llama" or "GPT" per joke

    # Scoring tracking (scores = Mistral normalized, used for ranking/threshold)
    scores: List[float]          # Mistral normalized 0-10 (primary ranking)
    final_scores: List[float]    # Ensemble GPT 60% + Mistral 40% (reference only)
    gpt_scores: List[float]      # GPT adjusted scores
    mistral_scores: List[float]  # Mistral raw 0-4

    # Best result tracking (selected by Mistral score)
    best_joke: str
    best_score: float

    # Iteration tracking
    iteration_count: int
    max_iterations: int

    # History for debugging
    history: List[Dict]


In [ ]:

# ============================================================================
# NODE FUNCTIONS
# ============================================================================

def generate_jokes_llama(state: JokeGenerationState, llama_gen: RAGHumorGenerator) -> Dict:
    """Generate jokes using Llama model"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    mode = "headline" if not word2 else "word-pair"
    print(f"\n[Iteration {iteration + 1}] Generating jokes with Llama ({mode} mode)...")

    try:
        result_with_RAG = llama_gen.generate_with_rag(
            word1=word1,
            word2=word2,
            use_rag=True,
            num_candidates=3
        )

        result_without_RAG = llama_gen.generate_with_rag(
            word1=word1,
            word2=word2,
            use_rag=False,
            num_candidates=3
        )

        jokes = result_with_RAG["generated_jokes"] + result_without_RAG["generated_jokes"]

        print('result_with_RAG jokes:')
        print(result_with_RAG["generated_jokes"])
        print('result_without_RAG jokes:')
        print(result_without_RAG["generated_jokes"])
        print(f"   Generated {len(jokes)} Llama jokes")
    except Exception as e:
        print(f"   Llama generation failed: {e}")
        jokes = []

    return {"llama_jokes": jokes}


def generate_jokes_gpt(state: JokeGenerationState, gpt_gen: GPT4JokeGenerator) -> Dict:
    """Generate jokes using GPT-OSS (reduced to 2 to limit self-bias)"""
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    mode = "headline" if not word2 else "word-pair"
    print(f"\n[Iteration {iteration + 1}] Generating jokes with GPT-OSS ({mode} mode)...")

    try:
        jokes = gpt_gen.generate_jokes(word1, word2, num_jokes=2, verbose=False)
        print(f"   Generated {len(jokes)} GPT-OSS jokes")
    except Exception as e:
        print(f"   GPT-OSS generation failed: {e}")
        jokes = []

    return {"gpt_jokes": jokes}


def combine_jokes(state: JokeGenerationState) -> JokeGenerationState:
    """Combine jokes from both generators and track sources"""
    llama_jokes = state.get("llama_jokes", [])
    gpt_jokes = state.get("gpt_jokes", [])

    all_jokes = llama_jokes + gpt_jokes
    sources = ["Llama"] * len(llama_jokes) + ["GPT"] * len(gpt_jokes)

    print(f"\nCombined {len(all_jokes)} total jokes ({len(llama_jokes)} Llama + {len(gpt_jokes)} GPT)")

    return {
        **state,
        "candidate_jokes": all_jokes,
        "joke_sources": sources
    }


def normalize_mistral_score(raw: float) -> float:
    """
    Normalize Mistral humor score (0-4) to 0-10 scale using
    percentile-aware mapping based on training data distribution.
    """
    raw = max(0.0, min(4.0, raw))
    
    breakpoints = [(0.0, 0.0), (1.0, 2.0), (2.0, 4.5), (2.26, 5.0),
                   (3.0, 7.0), (3.5, 8.5), (4.0, 10.0)]
    
    for i in range(len(breakpoints) - 1):
        x0, y0 = breakpoints[i]
        x1, y1 = breakpoints[i + 1]
        if raw <= x1:
            t = (raw - x0) / (x1 - x0)
            return y0 + t * (y1 - y0)
    
    return 10.0


def score_jokes_dual(
    state: JokeGenerationState,
    llm_scorer: LLMHumorScorer,
    mistral_scorer: MistralHumorScorer
) -> JokeGenerationState:
    """
    Score jokes using MISTRAL ONLY (fast mode - no LLM evaluation API call).
    LLM scorer is not called. LLM scores set to 0 (not available).
    """
    jokes = state["candidate_jokes"]
    sources = state.get("joke_sources", [])
    word1 = state["word1"]
    word2 = state["word2"]
    iteration = state.get("iteration_count", 0)

    if not jokes:
        print(f"\n[Iteration {iteration + 1}] No jokes to score, skipping...")
        return {
            **state,
            "scores": [], "final_scores": [], "gpt_scores": [], "mistral_scores": [],
            "best_joke": "", "best_score": 0.0
        }

    print(f"\n[Iteration {iteration + 1}] Scoring {len(jokes)} jokes with Mistral ONLY (fast mode, LLM eval skipped)...")

    # --- LLM evaluation SKIPPED for speed ---
    llm_scores = [0.0] * len(jokes)

    # --- Mistral: pure humor rating (0-4) ---
    print("   Scoring with Mistral (humor funniness only)...")
    mistral_scores = mistral_scorer.score_jokes(jokes, word1, word2, verbose=False)
    
    # Normalize Mistral to 0-10
    mistral_scores_normalized = [normalize_mistral_score(s) for s in mistral_scores]
    
    # No ensemble (LLM skipped) - final_scores = mistral_scores_normalized
    ensemble_scores = list(mistral_scores_normalized)

    # Print score breakdown sorted by Mistral
    print(f"\n   Score breakdown (MISTRAL ONLY - LLM eval skipped for speed):")
    print(f"   {'='*100}")
    print(f"   {'#':>3}  {'Src':<6} {'Mistral':>7} {'(norm)':>7}  Joke")
    print(f"   {'-'*100}")
    
    display_order = sorted(range(len(jokes)), key=lambda i: mistral_scores_normalized[i], reverse=True)
    for rank, i in enumerate(display_order, 1):
        joke = jokes[i]
        m_raw = mistral_scores[i]
        m_norm = mistral_scores_normalized[i]
        src = sources[i] if i < len(sources) else "?"
        marker = " << BEST" if rank == 1 else ""
        print(f"   {rank:3d}  {src:<6} {m_raw:5.2f}/4 ({m_norm:5.2f}/10)  {joke[:65]}...{marker}")

    # Find best joke by Mistral
    if mistral_scores_normalized:
        best_idx = mistral_scores_normalized.index(max(mistral_scores_normalized))
        best_joke = jokes[best_idx]
        best_score = mistral_scores_normalized[best_idx]
    else:
        best_joke = ""
        best_score = 0.0

    print(f"\n   Best: {best_score:.2f}/10  (raw: {mistral_scores[best_idx]:.2f}/4)")
    print(f"   Avg:  {sum(mistral_scores_normalized)/len(mistral_scores_normalized):.2f}/10")
    print(f"   Range: {min(mistral_scores_normalized):.2f} - {max(mistral_scores_normalized):.2f}")

    return {
        **state,
        "scores": mistral_scores_normalized,
        "final_scores": ensemble_scores,
        "gpt_scores": llm_scores,
        "mistral_scores": mistral_scores,
        "best_joke": best_joke,
        "best_score": best_score
    }


def check_score_threshold(state: JokeGenerationState) -> Literal["refine", "output"]:
    """Decision node: check if Mistral score meets threshold"""
    score = state["best_score"]
    iteration = state.get("iteration_count", 0)
    max_iter = state.get("max_iterations", 3)

    print(f"\nDecision: Mistral={score:.2f}/10, Iteration={iteration + 1}/{max_iter}")

    if score >= 7.0:
        print(f"   Mistral threshold met! (>=7.0 = raw 3.0/4)")
        return "output"
    elif iteration >= max_iter - 1:
        print(f"   Max iterations reached, outputting best result")
        return "output"
    else:
        print(f"   Mistral below threshold, refining...")
        return "refine"


def increment_iteration(state: JokeGenerationState) -> JokeGenerationState:
    """Increment iteration counter and save history"""
    current_iter = state.get("iteration_count", 0)
    history = state.get("history", [])

    history.append({
        "iteration": current_iter + 1,
        "best_score": state["best_score"],
        "best_joke": state["best_joke"],
        "num_candidates": len(state.get("candidate_jokes", [])),
        "avg_mistral_norm": sum(state.get("scores", [0])) / max(len(state.get("scores", [1])), 1),
        "avg_mistral_raw": sum(state.get("mistral_scores", [0])) / max(len(state.get("mistral_scores", [1])), 1)
    })

    return {
        **state,
        "iteration_count": current_iter + 1,
        "history": history
    }


def output_result(state: JokeGenerationState) -> JokeGenerationState:
    """Final output node - shows best and second-best joke ranked by Mistral"""
    print(f"\n{'='*80}")
    print(f"FINAL RESULT (Mistral only)")
    print(f"{'='*80}")

    word1 = state['word1']
    word2 = state['word2']
    if word2:
        print(f"\nWord Pair: '{word1}' + '{word2}'")
    else:
        print(f"\nHeadline: '{word1}'")
    print(f"Total Iterations: {state.get('iteration_count', 0) + 1}")

    jokes = state.get("candidate_jokes", [])
    m_norm = state.get("scores", [])
    mis_sc = state.get("mistral_scores", [])
    sources = state.get("joke_sources", [])

    if jokes and m_norm:
        ranked = sorted(
            zip(jokes, m_norm, mis_sc, sources + ["?"] * max(0, len(jokes) - len(sources))),
            key=lambda x: x[1], reverse=True
        )

        j1, mn1, m1, s1 = ranked[0]
        print(f"\n  #1 BEST  [Mistral:{mn1:.2f}/10 raw:{m1:.2f}/4]  Source:{s1}")
        print(f"  {j1}")

        if len(ranked) >= 2:
            j2, mn2, m2, s2 = ranked[1]
            print(f"\n  #2       [Mistral:{mn2:.2f}/10 raw:{m2:.2f}/4]  Source:{s2}")
            print(f"  {j2}")
    else:
        print(f"\n  (no jokes generated)")

    print(f"\n{'='*80}\n")

    return state

In [ ]:

# ============================================================================
# GRAPH BUILDER
# ============================================================================

def build_joke_generation_graph(
    llama_generator: RAGHumorGenerator,
    gpt_generator: GPT4JokeGenerator,
    llm_scorer: LLMHumorScorer,
    mistral_scorer: MistralHumorScorer
):
    """
    Build the LangGraph for joke generation pipeline.

    Args:
        llama_generator: RAGHumorGenerator
        gpt_generator: GPT4JokeGenerator
        llm_scorer: LLMHumorScorer (not called in headline mode)
        mistral_scorer: MistralHumorScorer (primary ranking)

    Returns:
        Compiled LangGraph
    """
    workflow = StateGraph(JokeGenerationState)

    # Add nodes with bound generators/scorers
    workflow.add_node(
        "start",
        lambda state: state  # pass-through to fan out to both generators
    )
    workflow.add_node(
        "generate_llama",
        lambda state: generate_jokes_llama(state, llama_generator)
    )
    workflow.add_node(
        "generate_gpt",
        lambda state: generate_jokes_gpt(state, gpt_generator)
    )
    workflow.add_node("combine_jokes", combine_jokes)
    workflow.add_node(
        "score_jokes",
        lambda state: score_jokes_dual(state, llm_scorer, mistral_scorer)
    )
    workflow.add_node("increment_iteration", increment_iteration)
    workflow.add_node("output", output_result)

    # Fan-out: start node dispatches to both generators in parallel
    workflow.set_entry_point("start")
    workflow.add_edge("start", "generate_llama")
    workflow.add_edge("start", "generate_gpt")

    # Add edges - both generators feed into combine
    workflow.add_edge("generate_llama", "combine_jokes")
    workflow.add_edge("generate_gpt", "combine_jokes")

    # After combining, score the jokes
    workflow.add_edge("combine_jokes", "score_jokes")

    # Decision point: refine or output
    workflow.add_conditional_edges(
        "score_jokes",
        check_score_threshold,
        {
            "refine": "increment_iteration",
            "output": "output"
        }
    )

    # Refinement loop - go back to start (fans out to both generators)
    workflow.add_edge("increment_iteration", "start")

    # End the workflow
    workflow.add_edge("output", END)

    return workflow.compile()

In [ ]:
# ============================================================================
# LOAD MODELS (run once, then skip when re-running pipeline)
# ============================================================================

print("="*80)
print("LANGGRAPH HEADLINE JOKE PIPELINE")
print("Mistral primary ranking | LLM eval skipped (headline fast mode)")
print("="*80)
print()

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found in environment. Please set it in .env file")

config = RAGConfig()
config.TEMPERATURE = 0.65
config.TOP_K = 50

print("Initializing components...\n")

# 1. Llama Generator
try:
    llama_gen = RAGHumorGenerator(config)
    llama_gen.load_model()
    print("Llama generator initialized\n")
except Exception as e:
    print(f"Could not load Llama model: {e}\n")
    llama_gen = None

# 2. GPT-OSS Generator
gpt_gen = GPT4JokeGenerator(
    api_key=NVIDIA_API_KEY,
    temperature=0.8,
    max_retries=1
)
print("GPT-OSS generator initialized\n")

# 3. LLM Scorer (not used in headline mode, but needed for graph signature)
llm_scorer = LLMHumorScorer(
    api_key=NVIDIA_API_KEY,
    temperature=0.4,
    max_retries=2
)
print("LLM scorer initialized (NOT used in headline mode)\n")

# 4. Mistral Scorer (primary ranking - fine-tuned on human humor ratings)
mistral_scorer = MistralHumorScorer(
    base_model_id="mistralai/Ministral-3-8B-Base-2512",
    num_labels=2,
    max_length=512
)
print(f"Mistral checkpoint: {mistral_scorer.checkpoint_path}")

if not mistral_scorer.checkpoint_path.exists():
    print(f"Mistral checkpoint not found! Expected: {mistral_scorer.checkpoint_path}")
    mistral_scorer = None
else:
    mistral_scorer.load_model()
    print("Mistral scorer initialized (PRIMARY)\n")

print("="*80)
print("All models loaded.")
print("="*80)

In [ ]:
# ============================================================================
# BUILD RAG DATABASE & COMPILE GRAPH (run once after loading models)
# ============================================================================

# Load RAG database
if llama_gen is not None:
    llama_gen.load_rag_database(build_if_missing=True)
    print("RAG database loaded\n")

# Build the graph
if mistral_scorer is None:
    raise RuntimeError("Cannot run pipeline without Mistral scorer (primary)")
if llama_gen is None:
    raise RuntimeError("Cannot run pipeline without Llama generator")

graph = build_joke_generation_graph(llama_gen, gpt_gen, llm_scorer, mistral_scorer)
print("LangGraph compiled (Mistral only, LLM eval skipped in headline mode)")

In [ ]:
# ============================================================================
# RUN HEADLINE PIPELINE
# Reads headlines from task-a-en.tsv, generates jokes, scores with Mistral only
# ============================================================================
import csv
import pandas as pd
from datetime import datetime

# Load headlines from task-a-en.tsv
tsv_path = PROJECT_ROOT / "data" / "task-a-en.tsv"
df_headlines = pd.read_csv(tsv_path, sep="\t")
print(f"Loaded {len(df_headlines)} headlines from {tsv_path.name}")

# Each row: id, word1(=-), word2(=-), headline
headlines = [(row["id"], row["headline"]) for _, row in df_headlines.iterrows()]

results_summary = []

# --- Open CSVs BEFORE the loop so results are saved incrementally ---
output_dir = PROJECT_ROOT / "data" / "headline_output"
output_dir.mkdir(exist_ok=True, parents=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

best_csv_path = output_dir / f"best_headline_jokes_{timestamp}.csv"
all_csv_path  = output_dir / f"all_headline_jokes_{timestamp}.csv"

best_csv_file = open(best_csv_path, 'w', newline='', encoding='utf-8')
all_csv_file  = open(all_csv_path,  'w', newline='', encoding='utf-8')

best_writer = csv.writer(best_csv_file)
all_writer  = csv.writer(all_csv_file)

best_writer.writerow(["#", "ID", "Headline", "Best_Joke", "Best_Source",
                       "2nd_Best_Joke", "2nd_Source",
                       "Best_Mistral_0to4", "Best_Mistral_Norm",
                       "2nd_Mistral_0to4",  "2nd_Mistral_Norm"])
all_writer.writerow(["#", "HeadlineIdx", "ID", "Rank", "Headline", "Joke",
                      "Source", "Mistral_0to4", "Mistral_0to10"])

all_row_num = 1
print(f"Incremental CSVs: {best_csv_path.name} / {all_csv_path.name}")
print(f"Scoring: Mistral ONLY (GPT evaluation skipped for speed)")

for h_idx, (h_id, headline_text) in enumerate(headlines, 1):
    print("\n" + "="*80)
    print(f"HEADLINE [{h_idx}/{len(headlines)}] {h_id}: {headline_text[:70]}...")
    print("="*80)

    # Headline mode: word1 = headline text, word2 = ""
    initial_state = {
        "word1": headline_text, "word2": "",
        "llama_jokes": [], "gpt_jokes": [], "candidate_jokes": [],
        "joke_sources": [],
        "scores": [], "final_scores": [], "gpt_scores": [], "mistral_scores": [],
        "best_joke": "", "best_score": 0.0,
        "iteration_count": 0, "max_iterations": 1, "history": []
    }

    try:
        result = graph.invoke(initial_state)
        total_candidates = len(result.get('candidate_jokes', []))
        sources = result.get('joke_sources', [])

        r = {
            "id": h_id,
            "headline": headline_text,
            "word1": headline_text,
            "word2": "",
            "word_pair": headline_text[:60],
            "best_score": result['best_score'],
            "best_joke": result['best_joke'],
            "all_jokes": result.get('candidate_jokes', []),
            "joke_sources": sources,
            "all_scores": result.get('scores', []),            # Mistral normalized
            "mistral_scores": result.get('mistral_scores', []),
            "iterations": result.get('iteration_count', 0) + 1,
            "total_candidates": total_candidates
        }
        results_summary.append(r)

        # --- Write to CSVs immediately ---
        jd = list(zip(r['all_jokes'], r['all_scores'], r['mistral_scores'],
                       sources + ["?"] * len(r['all_jokes'])))
        # Sort by Mistral normalized
        jd.sort(key=lambda x: x[1], reverse=True)

        if len(jd) >= 2:
            j1, mn1, m1, src1 = jd[0]
            j2, mn2, m2, src2 = jd[1]
        elif len(jd) == 1:
            j1, mn1, m1, src1 = jd[0]
            j2, mn2, m2, src2 = "", 0, 0, ""
        else:
            j1, mn1, m1, src1 = "", 0, 0, ""
            j2, mn2, m2, src2 = "", 0, 0, ""

        best_writer.writerow([
            h_idx, h_id, headline_text,
            j1, src1, j2, src2,
            f"{m1:.2f}", f"{mn1:.2f}",
            f"{m2:.2f}", f"{mn2:.2f}"
        ])
        best_csv_file.flush()

        for rank, (joke, m_norm, mistral, src) in enumerate(jd, 1):
            all_writer.writerow([
                all_row_num, h_idx, h_id, rank, headline_text, joke,
                src, f"{mistral:.2f}", f"{m_norm:.2f}"
            ])
            all_row_num += 1
        all_csv_file.flush()

    except Exception as e:
        print(f"\nError processing headline {h_id}: {e}")
        import traceback
        traceback.print_exc()

best_csv_file.close()
all_csv_file.close()

print(f"\nDone! Generated jokes for {len(results_summary)}/{len(headlines)} headlines.")
print(f"Best CSV : {best_csv_path}")
print(f"All CSV  : {all_csv_path}  ({all_row_num - 1} jokes)")

In [ ]:
# ============================================================================
# FULL OUTPUT FOR HUMAN RATING (re-run to reformat without regenerating)
# ============================================================================

print("#"*80)
print("#" + " "*22 + "HEADLINE JOKES FOR HUMAN RATING" + " "*25 + "#")
print("#"*80)
print(f"\nTotal headlines: {len(results_summary)}")
print(f"Total jokes generated: {sum(r['total_candidates'] for r in results_summary)}")
print(f"Scoring: MISTRAL ONLY (GPT evaluation skipped)")
print()

for h_idx, r in enumerate(results_summary, 1):
    print("="*80)
    print(f"HEADLINE {h_idx}/{len(results_summary)} [{r['id']}]: {r['headline'][:70]}")
    print(f"Best Mistral Score: {r['best_score']:.2f}/10")
    print("="*80)

    sources = r.get('joke_sources', [])
    joke_data = list(zip(
        r['all_jokes'], r['all_scores'], r['mistral_scores'],
        sources + ["?"] * len(r['all_jokes'])
    ))
    joke_data.sort(key=lambda x: x[1], reverse=True)

    for j_idx, (joke, m_norm, mistral, src) in enumerate(joke_data, 1):
        is_best = " << SELECTED" if joke == r['best_joke'] else ""
        print(f"\n  [{j_idx:2d}] Mistral: {m_norm:.2f}/10 (raw:{mistral:.2f}/4)  Src:{src}{is_best}")
        print(f"  {'-'*70}")
        print(f"  {joke}")

    print()

# --- Summary table ---
print("="*80)
print("RESULTS SUMMARY (Mistral only)")
print("="*80)
print(f"\n{'#':<4} {'ID':<12} {'Headline':<50} {'Mistral':<8} {'N':<4} {'BestSrc':<8}")
print("-"*90)
for i, r in enumerate(results_summary, 1):
    hl = r['headline'][:48]
    sources = r.get('joke_sources', [])
    best_src = "?"
    if r['all_jokes'] and r['all_scores'] and sources:
        best_idx = r['all_scores'].index(max(r['all_scores']))
        best_src = sources[best_idx] if best_idx < len(sources) else "?"
    print(f"{i:<4} {r['id']:<12} {hl:<50} {r['best_score']:<8.2f} {r['total_candidates']:<4} {best_src:<8}")

avg_score = sum(r['best_score'] for r in results_summary) / len(results_summary) if results_summary else 0
print("-"*90)
print(f"{'':4} {'':12} {'AVERAGE':<50} {avg_score:<8.2f}")

# --- Save JSON ---
json_path = output_dir / f"headline_jokes_{timestamp}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print(f"\nJSON saved: {json_path}")
print(f"CSVs saved incrementally:")
print(f"  Best    : {best_csv_path}")
print(f"  All jokes: {all_csv_path}")
print(f"\n{'='*80}")
print("Headline pipeline complete!")
print("="*80)